# 05 - PTQ with Tooling
Now that we've completed our objectives with virtual quantization, we'll use proper tooling to fully quantize the entire model.

We'll produce practical INT8/INT4 artifacts and inspect what the toolchain records so we can explain what it's doing, not just run it.  These outputs will be the inputs for the benchmarking notebook.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import platform
import json
import time

In [ ]:
MODEL_ID = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
PROJECT_ROOT = Path.cwd().resolve().parent
EXPORTS_DIR = PROJECT_ROOT / "exports"
RESULTS_DIR = PROJECT_ROOT / "results"

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model ID: {MODEL_ID}")
print(f"Platform: {platform.platform()}")
print(f"Exports dir: {EXPORTS_DIR}")
print(f"Results dir: {RESULTS_DIR}")

Model ID: deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct
Platform: macOS-15.7.3-arm64-arm-64bit
Exports dir: /Users/jarrett/dev/quantization-study/exports
Results dir: /Users/jarrett/dev/quantization-study/results


In [2]:
SOURCE_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-F16.gguf"

INT8_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf"
INT4_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"

print("SOURCE_GGUF:", SOURCE_GGUF)
print("INT8_GGUF:", INT8_GGUF)
print("INT4_GGUF:", INT4_GGUF)
print("source exists?", SOURCE_GGUF.exists())

SOURCE_GGUF: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf
INT8_GGUF: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf
INT4_GGUF: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
source exists? False


Convert cached Hugging Face weights to a GGUF source artifact

In [ ]:
SOURCE_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-F16.gguf"
Q8_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf"

cmd = [
    "llama-quantize",
    str(SOURCE_GGUF),
    str(Q8_GGUF),
    "Q8_0",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Done:", Q8_GGUF)
print("size GB:", round(Q8_GGUF.stat().st_size / 1e9, 2))

Running: llama-quantize /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf Q8_0


main: build = 8640 (7992aa7c8)
main: built with AppleClang 17.0.0.17000604 for Darwin arm64
main: quantizing '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf' to '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 52 key-value pairs and 404 tensors from /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = deepseek2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.300000
llama


main: quantize time = 17286.65 ms
main:    total time = 17286.65 ms
Done: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf
size GB: 16.7


llama_model_quantize_impl: model size  = 29964.48 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  = 15924.95 MiB (8.51 BPW)


Step: Produce INT4 artifact (Q4_K_M) from the same F16 GGUF source.
This gives us a lower-precision comparison point for Notebook 06 benchmarking (quality/speed/memory tradeoff vs Q8_0).           

In [ ]:
SOURCE_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-F16.gguf"
Q4_GGUF = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"

cmd = [
    "llama-quantize",
    str(SOURCE_GGUF),
    str(Q4_GGUF),
    "Q4_K_M",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Done:", Q4_GGUF)
print("size GB:", round(Q4_GGUF.stat().st_size / 1e9, 2))

Running: llama-quantize /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf Q4_K_M


main: build = 8640 (7992aa7c8)
main: built with AppleClang 17.0.0.17000604 for Darwin arm64
main: quantizing '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf' to '/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 52 key-value pairs and 404 tensors from /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = deepseek2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.300000
l


main: quantize time = 48554.26 ms
main:    total time = 48554.26 ms
Done: /Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
size GB: 10.37


size =   352.00 MiB ->    99.00 MiB
[ 404/ 404] blk.26.ffn_up_shexp.weight           - [  2048,   2816,      1,      1], type =    f16, converting to q4_K .. size =    11.00 MiB ->     3.09 MiB
llama_model_quantize_impl: model size  = 29964.48 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  =  9883.84 MiB (5.28 BPW)
llama_model_quantize_impl: WARNING: 54 of 404 tensor(s) required fallback quantization


Step: Save PTQ artifact summary for downstream benchmarking.
This records exact file sizes and compression ratios for F16, Q8_0, and Q4_K_M.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

source = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-F16.gguf"
q8 = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf"
q4 = EXPORTS_DIR / "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"

def file_info(p: Path):
    return {
        "path": str(p),
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else None,
        "size_gb_decimal": round(p.stat().st_size / 1e9, 2) if p.exists() else None,
        "size_gib_binary": round(p.stat().st_size / (1024**3),2) if p.exists() else None,
    }

report = {
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    "model_id": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    "ptq_method": "llama.cpp gguf quantization",
    "artifacts": {
        "f16_source": file_info(source),
        "q8_0": file_info(q8),
        "q4_k_m": file_info(q4),
    },
}

if source.exists() and q8.exists():
    report["compression_q8_vs_f16"] = round(q8.stat().st_size / source.stat().st_size, 4)
if source.exists() and q4.exists():
    report["compression_q4_vs_f16"] = round(q4.stat().st_size / source.stat().st_size, 4)

out_path = RESULTS_DIR / "05_ptq_artifacts_summary.json"
out_path.write_text(json.dumps(report, indent=2))
print(f"Wrote: {out_path}")
print(json.dumps(report, indent=2)[:1500])

Wrote: /Users/jarrett/dev/quantization-study/results/05_ptq_artifacts_summary.json
{
  "timestamp_utc": "2026-04-04T08:33:30.697898Z",
  "model_id": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
  "ptq_method": "llama.cpp gguf quantization",
  "artifacts": {
    "f16_source": {
      "path": "/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-F16.gguf",
      "exists": true,
      "size_bytes": 31424035968,
      "size_gb_decimal": 31.42,
      "size_gib_binary": 29.27
    },
    "q8_0": {
      "path": "/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q8_0.gguf",
      "exists": true,
      "size_bytes": 16702520448,
      "size_gb_decimal": 16.7,
      "size_gib_binary": 15.56
    },
    "q4_k_m": {
      "path": "/Users/jarrett/dev/quantization-study/exports/DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf",
      "exists": true,
      "size_bytes": 10367958144,
      "size_gb_decimal": 10.37,
      "size_gib_binary": 9.66
    }
  

/var/folders/1l/ts95kt9945j5dtyz_yqmlmpw0000gn/T/ipykernel_5171/3308780651.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",
